# IMDb archived title pages via Tor

Fetches the Wayback Machine capture list for each IMDb ID in a CSV, then downloads the latest scrapable capture with Selenium through Tor.

Dependencies: `python -m pip install "requests[socks]" selenium`

In [1]:
import csv
import json
import random
import re
import socket
import time
from pathlib import Path
from urllib.parse import urlencode

import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# ---- Settings (replaces the command line args) ----
CSV_PATH = Path("imdb_ids_with_titles.csv")  # CSV with an imdb_id column
LIMIT = 0                                    # number of IDs to scrape, 0 = all
RESPONSES_DIR = Path("responses")
DESCRIPTIONS_DIR = Path("descriptions")

CDX_URL = "https://web.archive.org/cdx/search/cdx"
TOR_HOST, TOR_PORT = "127.0.0.1", 9050
PROXIES = {
    "http": f"socks5h://{TOR_HOST}:{TOR_PORT}",
    "https": f"socks5h://{TOR_HOST}:{TOR_PORT}",
}

RESPONSES_DIR.mkdir(parents=True, exist_ok=True)
DESCRIPTIONS_DIR.mkdir(parents=True, exist_ok=True)

try:
    with socket.create_connection((TOR_HOST, TOR_PORT), timeout=5):
        print("proxy port is reachable")
except OSError as e:
    print(f"proxy connection failed: {e}")

proxy port is reachable


In [2]:
def movie_ids(csv_path):
    with csv_path.open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        if "imdb_id" not in (reader.fieldnames or []):
            raise ValueError(f"{csv_path} must contain an 'imdb_id' column")

        seen = set()
        for row in reader:
            imdb_id = (row.get("imdb_id") or "").strip()
            if not re.fullmatch(r"tt\d+", imdb_id):
                raise ValueError(f"Invalid IMDb ID in CSV: {imdb_id!r}")
            if imdb_id not in seen:
                seen.add(imdb_id)
                yield imdb_id


def fetch_captures(imdb_id, responses_dir):
    params = urlencode({
        "url": f"https://www.imdb.com/title/{imdb_id}",
        "output": "json",
        "fl": "timestamp,original,statuscode,mimetype,digest",
        "filter": ["statuscode:200", "mimetype:text/html"],
    }, doseq=True)
    response = requests.get(f"{CDX_URL}?{params}", proxies=PROXIES, timeout=60)
    response.raise_for_status()

    cdx_data = response.content
    (responses_dir / f"response_{imdb_id}.txt").write_bytes(cdx_data)

    captures = json.loads(cdx_data)
    if len(captures) < 2:
        raise RuntimeError(f"No archived HTML captures found for {imdb_id}")
    return captures[1:]  # drop the header row

In [3]:
def getPage(url, folder, exact_date, page_name, group_name, file_extension):

    output_dir = Path(folder)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{exact_date}_{page_name}.{file_extension}"

    if output_file.exists():
        print(f"SKIP {exact_date}: Already downloaded")
        return True

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--proxy-server=socks5://127.0.0.1:9050")

    driver = webdriver.Chrome(options=options)

    try:
        driver.set_page_load_timeout(90)

        for attempt in range(3):
            try:
                driver.get("https://check.torproject.org/")
                if (driver.title == "Congratulations. This browser is configured to use Tor."
                        and driver.current_url == "https://check.torproject.org/"):
                    print("__TOR is OK")
                driver.get(url)
                time.sleep(3)

                html = driver.page_source

                if (
                    "403 Forbidden" in html
                    or "Error 403" in html
                    or "ERR_TOO_MANY_REDIRECTS" in html
                    or "redirected you too many times" in html
                    or "This site can’t be reached" in html
                ):
                    print(f"NOT SCRAPPABLE: {exact_date}")
                    return False

                if "429 Too Many Requests" in html:
                    print(f"429 RATE LIMIT: {exact_date}")
                    return None

                if len(html) < 1000:
                    raise Exception("Page too short")

                output_file.write_text(html, encoding="utf-8")

                print(f"OK {exact_date}: {output_file}")
                return True

            except Exception as e:
                print(f"Attempt {attempt + 1}/3 failed: {exact_date}: {e}")

                if attempt < 2:
                    time.sleep(10)

        return False

    finally:
        driver.quit()

In [4]:

def scrape_movie(imdb_id):
    captures = fetch_captures(imdb_id, RESPONSES_DIR)
    # Try captures from latest to oldest, stop at the first one that works
    for timestamp, original, *_ in sorted(captures, key=lambda row: row[0], reverse=True):

        success = getPage(
            f"https://web.archive.org/web/{timestamp}id_/{original}",
            DESCRIPTIONS_DIR,
            timestamp,
            imdb_id,
            imdb_id,
            "html",
        )
        if success:
            return True
        sleep_time = random.uniform(1, 4)
        time.sleep(sleep_time)

    print(f"FAILED {imdb_id}: no archived page could be downloaded "
          f"after trying {len(captures)} captures")
    return False


ids = movie_ids(CSV_PATH)
selected_ids = ids if LIMIT == 0 else (i for n, i in enumerate(ids) if n < LIMIT)
count = 0
for imdb_id in selected_ids:
    count = count+1
    print(count)
    print(f"\n__________________ {imdb_id} __________________\n")
    try:
        scrape_movie(imdb_id)
    except Exception as e:
        print(f"ERROR {imdb_id}: {e}")

1

__________________ tt0068646 __________________

SKIP 20261118042323: Already downloaded
2

__________________ tt0167260 __________________

SKIP 20261118042323: Already downloaded
3

__________________ tt0111161 __________________

SKIP 20261118042323: Already downloaded
4

__________________ tt0071562 __________________

SKIP 20261118042323: Already downloaded
5

__________________ tt0047478 __________________

SKIP 20261118042326: Already downloaded
6

__________________ tt0108052 __________________

SKIP 20261118042323: Already downloaded
7

__________________ tt0034583 __________________

SKIP 20261118042329: Already downloaded
8

__________________ tt0120737 __________________

SKIP 20261118064920: Already downloaded
9

__________________ tt0167261 __________________

SKIP 20261118042324: Already downloaded
10

__________________ tt0076759 __________________

SKIP 20261118042327: Already downloaded
11

__________________ tt0033467 __________________

SKIP 20261118042347: Alrea